## 1. Clarify the Game Constraints

Before touching models, lock in the constraints.

* **Task:** Solve olympiad‑level LaTeX math problems, output an integer in a fixed range (e.g., 0–99999).
* **Environment:** Kaggle Notebook (No Internet, strict time limit). You must use the official evaluation API.
* **Compute:**
* *Training:* External H100 cluster (Fields initiative/cloud).
* *Inference:* Kaggle Notebook (CPU/GPU) using uploaded weights.


* **Judging:** Accuracy over public + private problem sets. Instability is penalized.

> **Your Mental Model:** Train anywhere, but the final model must run **fully offline** and **deterministically** inside a Kaggle notebook.

---

## 2. Set Up a Minimal Baseline

**Goal:** A completely working system, even if it’s "dumb."

1. **Clone a Baseline Solver:**
* Look for "AIMO Efficient Local Model Solver" or similar on Kaggle.


2. **Fork into your own Notebook:**
* **Keep:** Evaluation API usage, main inference loop, answer serialization.
* **Replace:** The toy model (later) and simple reasoning logic.


3. **Connect to GitHub:**
* `data/`: Training + Validation sets.
* `training/`: Fine-tuning scripts.
* `inference/`: Kaggle notebook templates and configs.



*Treat the Kaggle notebook as the "deployment" target of your GitHub repo.*

---

## 3. Design Data & Reasoning Format

Your GitHub data is where your model becomes "you."

### 3.1 Decide Problem → Target Format

For each problem, store a structure like:

* **Inputs:** `problem_latex`, `topic`, `difficulty`
* **Outputs (Training):** `solution_steps` (Chain-of-Thought), `code_solution` (optional), `final_answer`

**Target Prompt Format:**

> **Prompt:** "You are a math problem solver… Problem: [Latex]..."
> **Output:** "Reasoning: [Steps] …  42"

### 3.2 Build a Clean Dataset Pipeline

Organize your repo folders:

* `data/raw/`: Original sources (AIME, AMC, AIMO).
* `data/processed/`: Standardized JSONL/Parquet.
* `data/val/`: Held-out problems (never train on these).

**Scripts needed:**

* Clean LaTeX.
* Normalize answer formats.
* Verify `final_answer` is an integer.
* Generate reasoning traces (manual or model-assisted).

---

## 4. Choose & Prepare Base Model

Pick a strong open model that fits your compute.

* **Candidates:** DeepSeekMath, Llama-3, Qwen, Mistral (Fine-tuned on math).
* **Constraints:**
* Must fit on H100s for training.
* Must fit in **Kaggle GPU/CPU memory** at inference (7B–14B is usually safer than 70B).


* **Config:** Create `config/model.yaml` (base model name, dtype, LoRA config).

---

## 5. Fine-Tune on H100 Compute

This is the heavy lifting.

### 5.1 Pick Method

* **QLoRA / LoRA:** Train low-rank adapters. Cheaper and usually sufficient.
* **Full Fine-Tuning:** Only if you have massive compute (DeepSpeed/FSDP).

### 5.2 Training Loop Structure

1. Load Base Model + Tokenizer.
2. Load Processed Dataset.
3. Apply LoRA.
4. Train (Log loss + eval on private validation set).
5. **Qualitative Check:** Periodically sample problems to inspect reasoning.

### 5.3 Export for Kaggle

* Save **merged weights** OR **base + adapter** (with a merge script).
* Compress and store locally/in bucket to upload as a Kaggle Dataset.

---

## 6. Build Inference Pipeline (Kaggle)

This is your "Deployment" phase. inside `inference/`:

1. **Model Loader:** Load weights from Kaggle dataset. Set to `eval` mode and **fix seeds** (determinism).
2. **Problem Handler:** ingest LaTeX from API -> Build Prompt.
3. **Reasoning Engine:**
* *Start:* Single Chain-of-Thought (CoT) pass.
* *Upgrade:* Self-Consistency (Majority vote), PAL (Python code execution), or Verifier models.


4. **Answer Extractor:** Parse output to find the integer. Handle corner cases (no number, nonsense).

---

## 7. Iterate: Eval, Error Analysis, Refine

This is where you win.

1. **Local Harness:** Run full pipeline on held-out data.
2. **Error Analysis:** Classify errors (Parsing? Arithmetic? Conceptual?).
3. **Refine:**
* Fix via Data (add examples).
* Fix via Inference (tweak temperature, top-p, verifiers).



---

## 8. Documentation & Next Steps

**Immediate To-Do List:**

1. [ ] **Lock Data Format:** Define JSONL schema for problems/solutions.
2. [ ] **Freeze Model:** Choose base (e.g., Qwen2.5 7B) + method (QLoRA).
3. [ ] **Training Script:** Set up arguments, data loader, logging.
4. [ ] **Kaggle Fork:** Create a dummy notebook that passes the evaluation loop.

**Next:** Tell me your **base model choice** and **current dataset size**, and I will draft the training stack code for you.